# Speech Emotion Recognition with Audio Flamingo 3 (NVIDIA)
Laedt das EMO-DB-Dataset direkt von Kaggle herunter und klassifiziert die Emotion zero-shot mit `nvidia/audio-flamingo-3-hf`.

**Lizenz-Hinweis**: Audio Flamingo 3 steht unter der NVIDIA OneWay Noncommercial License (nur fuer nicht-kommerzielle Forschung). Vergleiche https://huggingface.co/nvidia/audio-flamingo-3.

**Hardware**: 7B Parameter, A100/H100 empfohlen. Auf kleineren GPUs `device_map=auto` + `bfloat16`, ggf. Out-of-Memory bei langen Clips. EmoDB-Clips sind <10s, das passt.

In [1]:
# Audio Flamingo 3 ist in `transformers` integriert, braucht aber eine aktuelle Version (git main).
!pip install -q kaggle librosa
!pip install -q --upgrade "git+https://github.com/huggingface/transformers" accelerate


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.6.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os

# Kaggle-Zugangsdaten hier eintragen
os.environ["KAGGLE_USERNAME"] = "DEIN_USERNAME"
os.environ["KAGGLE_KEY"]      = "DEIN_API_KEY"

# Kaggle-Dataset-Slug (Teil der URL nach kaggle.com/datasets/)
DATASET_SLUG = "piyushagni5/berlin-database-of-emotional-speech-emodb"
DOWNLOAD_DIR = "/tmp/emodb"

# Prompt-Modus: 'normal' oder 'prosodic'
PROMPT_MODE = "normal"

# Dataset von Kaggle herunterladen und entpacken
!kaggle datasets download -d {DATASET_SLUG} -p {DOWNLOAD_DIR} --unzip

Dataset URL: https://www.kaggle.com/datasets/piyushagni5/berlin-database-of-emotional-speech-emodb
License(s): copyright-authors
100%|██████████████████████████████████████| 38.0M/38.0M [00:01<00:00, 22.8MB/s]



In [3]:
from pathlib import Path
all_files = sorted(Path(DOWNLOAD_DIR).rglob("*.wav"))
print(f"Gefundene WAV-Dateien: {len(all_files)}")
for f in all_files[:5]:
    print(f"  {f}")

Gefundene WAV-Dateien: 535
  /tmp/emodb/wav/03a01Fa.wav
  /tmp/emodb/wav/03a01Nc.wav
  /tmp/emodb/wav/03a01Wa.wav
  /tmp/emodb/wav/03a02Fc.wav
  /tmp/emodb/wav/03a02Nc.wav


In [4]:
import json
import torch
from transformers import AudioFlamingo3ForConditionalGeneration, AutoProcessor

PROMPTS = {
    "normal": (
        "Listen to this audio clip and classify the emotion of the speaker. "
        "Choose from: anger, anxiety, boredom, disgust, fear, happiness, neutral, sadness. "
        "Reply with just the emotion label."
    ),
    "prosodic": (
        "Listen to this audio clip and classify the emotion of the speaker "
        "based only on prosodic and acoustic features such as pitch, "
        "speech rate, rhythm and loudness. "
        "Ignore the spoken words and linguistic content entirely. "
        "Choose from: anger, anxiety, boredom, disgust, fear, happiness, neutral, sadness. "
        "Reply with just the emotion label."
    ),
}

PROMPT = PROMPTS[PROMPT_MODE]
output_file = f"emotion_results_audioflamingo_{PROMPT_MODE}.json"

# Resume
results = {}
if os.path.exists(output_file):
    with open(output_file) as f:
        results = json.load(f)
    print(f"{len(results)} Dateien bereits verarbeitet")

# Modell laden
print("Lade Modell...")
MODEL_ID = "nvidia/audio-flamingo-3-hf"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AudioFlamingo3ForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
).eval()
print(f"Modell geladen auf: {next(model.parameters()).device}")

Lade Modell...


processor_config.json:   0%|          | 0.00/494 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/703 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/16.5G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/830 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/145 [00:00<?, ?B/s]

Modell geladen auf: cuda:0


In [7]:
total_files = len(all_files)

# Audio Flamingo 3 akzeptiert Dateipfade direkt — kein vorheriges librosa-Loading noetig.
for i, audio_file in enumerate(all_files, 1):
    if audio_file.name in results:
        print(f"({i}/{total_files}) Uebersprungen: {audio_file.name}")
        continue

    print(f"({i}/{total_files}) {audio_file.name}", end=" ... ")
    try:
        conversation = [{
            "role": "user",
            "content": [
                {"type": "text",  "text": PROMPT},
                {"type": "audio", "path": str(audio_file)},
            ],
        }]

        inputs = processor.apply_chat_template(
            conversation,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
        )
        # Float-Tensoren (Audio-Features) muessen in das Modell-dtype (bf16)
        # gecastet werden; int-Tensoren (input_ids, attention_mask) bleiben.
        inputs = {
            k: (v.to(model.device, dtype=torch.bfloat16) if torch.is_tensor(v) and v.is_floating_point()
                else v.to(model.device) if torch.is_tensor(v)
                else v)
            for k, v in inputs.items()
        }

        with torch.no_grad():
            generate_ids = model.generate(**inputs, max_new_tokens=10, do_sample=False)
        generate_ids = generate_ids[:, inputs["input_ids"].shape[1]:]
        emotion = processor.batch_decode(generate_ids, skip_special_tokens=True)[0].strip()

        results[audio_file.name] = emotion
        print(emotion)

        with open(output_file, "w") as f:
            json.dump(results, f, indent=2)

    except Exception as e:
        print(f"FEHLER: {e}")

print(f"\nFertig. {len(results)}/{total_files} Dateien verarbeitet.")

(1/535) 03a01Fa.wav ... happy
(2/535) 03a01Nc.wav ... neutral
angry5) 03a01Wa.wav ... 
(4/535) 03a02Fc.wav ... happy
(5/535) 03a02Nc.wav ... neutral
(6/535) 03a02Ta.wav ... sad
angry5) 03a02Wb.wav ... 
(8/535) 03a02Wc.wav ... angry
(9/535) 03a04Ad.wav ... fear
happy35) 03a04Fd.wav ... 
(11/535) 03a04Lc.wav ... neutral
(12/535) 03a04Nc.wav ... neutral
(13/535) 03a04Ta.wav ... neutral
angry35) 03a04Wc.wav ... 
fear535) 03a05Aa.wav ... 
(16/535) 03a05Fc.wav ... happy
(17/535) 03a05Nd.wav ... neutral
(18/535) 03a05Tc.wav ... neutral
angry35) 03a05Wa.wav ... 
(20/535) 03a05Wb.wav ... angry
(21/535) 03a07Fa.wav ... happy
happy35) 03a07Fb.wav ... 
(23/535) 03a07La.wav ... neutral
(24/535) 03a07Nc.wav ... neutral
angry35) 03a07Wc.wav ... 
(26/535) 03b01Fa.wav ... disgust
(27/535) 03b01Lb.wav ... disgust
neutral) 03b01Nb.wav ... 
(29/535) 03b01Td.wav ... sad
(30/535) 03b01Wa.wav ... angry
angry35) 03b01Wc.wav ... 
(32/535) 03b02Aa.wav ... fear
(33/535) 03b02La.wav ... neutral
neutral) 03b02Na.w

In [6]:
import pandas as pd

df = pd.DataFrame(list(results.items()), columns=["file", "emotion"])
print(df["emotion"].value_counts().to_string())

csv_file = output_file.replace(".json", ".csv")
df.to_csv(csv_file, index=False)
print(f"\nCSV gespeichert: {csv_file}")

Series([], )

CSV gespeichert: emotion_results_audioflamingo_normal.csv
